# NOTEBOOK 1: XAI HEATMAP GENERATION FACTORY
# ISIC 2019 Skin Lesion Classification — Explainability Pipeline
# PURPOSE : Generate and save Grad-CAM++ (DenseNet, Swin) and Score-CAM (EfficientNet)
#           heatmaps for the validation/test set. Results are persisted as .npy files
#           so downstream notebooks never have to re-run GPU inference.

In [ ]:
!pip install grad-cam -q
!pip install timm -q


In [1]:
import os
import numpy as np

OUTPUT_DIR = '/kaggle/working/xai_outputs'
BINARY_OUT = os.path.join(OUTPUT_DIR, 'binary')
MULTI_OUT  = os.path.join(OUTPUT_DIR, 'multiclass')

print("=" * 50)
print("             XAI FACTORY AUDIT REPORT          ")
print("=" * 50)

# 1. Check File Counts
print("\n[1/3] Checking Saved Matrix Counts:")
binary_folders = ['heatmaps_densenet', 'heatmaps_efficientnet', 'heatmaps_swin', 'heatmaps_convnext', 'heatmaps_resnet', 'heatmaps_ensemble']
severity_folders = ['heatmaps_densenet', 'heatmaps_efficientnet', 'heatmaps_swin']

print("  ── Binary Run Output (Notebook 1 - Part 1) ──")
for folder in binary_folders:
    path = os.path.join(BINARY_OUT, folder)
    count = len([f for f in os.listdir(path) if f.endswith('.npy')]) if os.path.exists(path) else 0
    print(f"    {folder:<25}: {count} files expected ~5067")

print("\n  ── Multiclass Severity Output (Notebook 1 - Part 2) ──")
for folder in severity_folders:
    path = os.path.join(MULTI_OUT, folder)
    count = len([f for f in os.listdir(path) if f.endswith('.npy')]) if os.path.exists(path) else 0
    print(f"    {folder:<25}: {count} files expected ~9340")

# 2. Check Meta-Indices
print("\n[2/3] Checking Metadata Index CSVs:")
index_path = os.path.join(BINARY_OUT, 'heatmap_index.csv')
if os.path.exists(index_path):
    print(f"    ✓ heatmap_index.csv found at {index_path}")
    import pandas as pd
    df_check = pd.read_csv(index_path)
    print(f"    ✓ Registered mapped rows: {len(df_check)}")
else:
    print("    ✗ Critical Warning: heatmap_index.csv is missing from binary output.")

# 3. Structural Matrix Integrity Check
print("\n[3/3] Testing Random Matrix Integrity:")
try:
    # Pick a random sample from ensemble
    sample_dir = os.path.join(BINARY_OUT, 'heatmaps_ensemble')
    sample_file = [f for f in os.listdir(sample_dir) if f.endswith('.npy')][0]
    matrix = np.load(os.path.join(sample_dir, sample_file))
    
    print(f"    ✓ Loaded sample matrix: {sample_file}")
    print(f"    ✓ Spatial Dimensions   : {matrix.shape} (Should be 224x224 after fusion)")
    print(f"    ✓ Value Normalization  : Min={matrix.min():.2f}, Max={matrix.max():.2f} (Should be 0.00 to 1.00)")
    print("\n[STATUS]: Everything is completely intact. Safe to compress and transition downstream.")
except Exception as e:
    print(f"    ✗ Error testing file matrix integrity: {str(e)}")
print("=" * 50)

             XAI FACTORY AUDIT REPORT          

[1/3] Checking Saved Matrix Counts:
  ── Binary Run Output (Notebook 1 - Part 1) ──
    heatmaps_densenet        : 0 files expected ~5067
    heatmaps_efficientnet    : 0 files expected ~5067
    heatmaps_swin            : 0 files expected ~5067
    heatmaps_convnext        : 0 files expected ~5067
    heatmaps_resnet          : 0 files expected ~5067
    heatmaps_ensemble        : 0 files expected ~5067

  ── Multiclass Severity Output (Notebook 1 - Part 2) ──
    heatmaps_densenet        : 0 files expected ~9340
    heatmaps_efficientnet    : 0 files expected ~9340
    heatmaps_swin            : 0 files expected ~9340

[2/3] Checking Metadata Index CSVs:
    ✗ Critical Warning: heatmap_index.csv is missing from binary output.

[3/3] Testing Random Matrix Integrity:
    ✗ Error testing file matrix integrity: [Errno 2] No such file or directory: '/kaggle/working/xai_outputs/binary/heatmaps_ensemble'


In [2]:
# ── CELL 1 : Installs & Imports ──────────────────────────────────────────────
import os, gc, json, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from pathlib import Path
from PIL import Image
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import timm

warnings.filterwarnings('ignore')

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {DEVICE}")

Device : cuda


In [3]:
# ── CELL 2 : Configuration ────────────────────────────────────────────────────
# ── Input paths ──────────────────────────────────────────────────────────────
INPUT_CSV_DIR   = '/kaggle/input/notebooks/nandini143/data-preparation'
MODEL_DIR       = '/kaggle/input/notebooks/nandini143'

# ── Binary classification checkpoint paths ────────────────────────────────────
DENSENET_CKPT   = '/kaggle/input/notebooks/nandini143/binary-densenet-121/densenet121_best.pth'
EFFNET_CKPT     = '/kaggle/input/notebooks/nandini143/binary-efficientnetv2s/efficientnetv2s_best.pth'
SWIN_CKPT       = '/kaggle/input/notebooks/nandini143/binary-swin-tiny-transformer/swin_tiny_best.pth'
CONVNEXT_CKPT   = '/kaggle/input/notebooks/nandini143/binary-convnext/convnext_tiny_best.pth'
RESNET_CKPT     = '/kaggle/input/notebooks/nandini143/binary-resnet-200d/resnet200d_best.pth'

# ── Multiclass (Severity) checkpoint paths ────────────────────────────────────
SEVERITY_DENSENET_CKPT  = '/kaggle/input/notebooks/nandini143/densenet-121-ordinal-severity/densenet_severity/densenet_severity_best.pth'
SEVERITY_EFFNET_CKPT    = '/kaggle/input/notebooks/nandini143/efficientnetv2-ordinal-severity/effnet_severity/effnet_severity_best.pth'
SEVERITY_SWIN_CKPT      = '/kaggle/input/notebooks/nandini143/swin-tiny-ordinal-severity/swin_severity/swin_severity_best.pth'

# ── Multiclass severity CSV (for image IDs & metadata) ───────────────────────
SEVERITY_CSV    = '/kaggle/input/datasets/nandini143/multiclass-output/severity_data/severity_train_folds.csv'
SEVERITY_IMGDIR = '/kaggle/input/datasets/nandini143/multiclass-output/severity_data/preprocessed_images'

# ── CORAL config: 4 severity classes (AKIEC, BCC, SCC, MEL) → 3 binary outputs
SEVERITY_NUM_OUTPUTS = 3
SEVERITY_META_DIM    = 9    # 3 (age_norm, sex_male, sex_female) + 6 site dummies (typical ISIC)
STAGE_NAMES          = {0: 'AK', 1: 'BCC', 2: 'SCC', 3: 'MEL'}

# ── Image sizes ───────────────────────────────────────────────────────────────
IMG_SIZE_CNN    = 384
IMG_SIZE_SWIN   = 224
BATCH_SIZE      = 1    # MUST be 1 for accurate XAI generation

# ── Path replacement (for loading images from Kaggle datasets) ────────────────
OLD_PATH        = '/kaggle/working/preprocessed/'
NEW_PATH        = f'{INPUT_CSV_DIR}/preprocessed/'

# ── Normalization ─────────────────────────────────────────────────────────────
IMAGENET_MEAN   = [0.485, 0.456, 0.406]
IMAGENET_STD    = [0.229, 0.224, 0.225]

# ── Output directories: separated for binary vs multiclass heatmaps ───────────
OUTPUT_DIR      = '/kaggle/working/xai_outputs'
BINARY_OUT      = os.path.join(OUTPUT_DIR, 'binary')
MULTI_OUT       = os.path.join(OUTPUT_DIR, 'multiclass')

for sub in ['heatmaps_densenet', 'heatmaps_efficientnet', 'heatmaps_swin', 'heatmaps_convnext', 'heatmaps_resnet', 'heatmaps_ensemble']:
    os.makedirs(os.path.join(BINARY_OUT, sub), exist_ok=True)
    os.makedirs(os.path.join(MULTI_OUT, sub),  exist_ok=True)

os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Config ready.')
print(f'  Binary   outputs -> {BINARY_OUT}')
print(f'  Severity outputs -> {MULTI_OUT}')

Config ready.
  Binary   outputs -> /kaggle/working/xai_outputs/binary
  Severity outputs -> /kaggle/working/xai_outputs/multiclass


In [4]:
# ── CELL 3a : Binary Model Architectures (CORRECTED) ───────────────────────
class TabularMLP_ConvNeXt(torch.nn.Module):
    def __init__(self, in_dim, hidden_dim=32, out_dim=32, dropout=0.4):
        super().__init__()
        self.net = torch.nn.Sequential(
            torch.nn.Linear(in_dim, hidden_dim), torch.nn.BatchNorm1d(hidden_dim), torch.nn.GELU(),
            torch.nn.Dropout(dropout), torch.nn.Linear(hidden_dim, out_dim), torch.nn.GELU()
        )
    def forward(self, x): return self.net(x)

# ── FIXED: Added MultiModalWrappers for DenseNet and EfficientNet to match checkpoint ──
class MultiModalDenseNet(torch.nn.Module):
    def __init__(self, backbone_name='densenet121', meta_dim=11, pretrained=False):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, pretrained=pretrained, num_classes=0, global_pool='avg')
        img_feat_dim = self.backbone.num_features
        self.meta_mlp = TabularMLP_ConvNeXt(meta_dim, 32, 32, 0.4)
        fusion_dim = img_feat_dim + 32
        self.head = torch.nn.Sequential(
            torch.nn.Dropout(0.4), torch.nn.Linear(fusion_dim, 256), torch.nn.BatchNorm1d(256),
            torch.nn.GELU(), torch.nn.Dropout(0.2), torch.nn.Linear(256, 1)
        )
    def forward(self, images, meta=None):
        img_feats = self.backbone(images)
        meta_feats = self.meta_mlp(meta) if meta is not None else self.meta_mlp(torch.zeros(images.size(0), 11, device=images.device))
        return self.head(torch.cat([img_feats, meta_feats], dim=1))

class MultiModalEfficientNet(torch.nn.Module):
    def __init__(self, backbone_name='tf_efficientnetv2_s', meta_dim=11, pretrained=False):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, pretrained=pretrained, num_classes=0, global_pool='avg', drop_path_rate=0.2)
        img_feat_dim = self.backbone.num_features
        self.meta_mlp = TabularMLP_ConvNeXt(meta_dim, 32, 32, 0.4)
        fusion_dim = img_feat_dim + 32
        self.head = torch.nn.Sequential(
            torch.nn.Dropout(0.4), torch.nn.Linear(fusion_dim, 256), torch.nn.BatchNorm1d(256),
            torch.nn.GELU(), torch.nn.Dropout(0.2), torch.nn.Linear(256, 1)
        )
    def forward(self, images, meta=None):
        img_feats = self.backbone(images)
        meta_feats = self.meta_mlp(meta) if meta is not None else self.meta_mlp(torch.zeros(images.size(0), 11, device=images.device))
        return self.head(torch.cat([img_feats, meta_feats], dim=1))

class MultiModalConvNeXt(torch.nn.Module):
    def __init__(self, backbone_name='convnext_tiny', meta_dim=11, pretrained=False):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, pretrained=pretrained, num_classes=0)
        img_feat_dim = self.backbone.num_features
        self.meta_mlp = TabularMLP_ConvNeXt(meta_dim, 32, 32, 0.4)
        fusion_dim = img_feat_dim + 32
        self.head = torch.nn.Sequential(
            torch.nn.Dropout(0.4), torch.nn.Linear(fusion_dim, 256), torch.nn.BatchNorm1d(256),
            torch.nn.GELU(), torch.nn.Dropout(0.2), torch.nn.Linear(256, 1)
        )
    def forward(self, images, meta=None):
        img_feats = self.backbone(images)
        meta_feats = self.meta_mlp(meta) if meta is not None else self.meta_mlp(torch.zeros(images.size(0), 11, device=images.device))
        return self.head(torch.cat([img_feats, meta_feats], dim=1))

class MultiModalResNet(torch.nn.Module):
    def __init__(self, model_name='resnet200d', num_meta_features=11, pretrained=False):
        super().__init__()
        self.backbone = timm.create_model(model_name, pretrained=pretrained, num_classes=0)
        in_features = self.backbone.num_features
        self.meta_branch = torch.nn.Sequential(
            torch.nn.Linear(num_meta_features, 64), torch.nn.BatchNorm1d(64), torch.nn.SiLU(),
            torch.nn.Dropout(0.2), torch.nn.Linear(64, 32), torch.nn.BatchNorm1d(32), torch.nn.SiLU()
        )
        self.head = torch.nn.Sequential(
            torch.nn.Linear(in_features + 32, 256), torch.nn.BatchNorm1d(256), torch.nn.SiLU(),
            torch.nn.Dropout(0.4), torch.nn.Linear(256, 1)
        )
    def forward(self, image, meta=None):
        img_features = self.backbone(image)
        meta_features = self.meta_branch(meta) if meta is not None else self.meta_branch(torch.zeros(image.size(0), 11, device=image.device))
        return self.head(torch.cat([img_features, meta_features], dim=1))

class MultiModalSwin(torch.nn.Module):
    def __init__(self, backbone_name='swin_tiny_patch4_window7_224', meta_dim=11, pretrained=False):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, pretrained=pretrained, num_classes=0, drop_path_rate=0.2)
        img_feat_dim = self.backbone.num_features 
        self.meta_mlp = TabularMLP_ConvNeXt(meta_dim, 32, 32, 0.4)
        fusion_dim = img_feat_dim + 32 
        
        self.head = torch.nn.Sequential(
            torch.nn.Dropout(0.4), torch.nn.Linear(fusion_dim, 256), torch.nn.BatchNorm1d(256),
            torch.nn.GELU(), torch.nn.Dropout(0.2), torch.nn.Linear(256, 1) 
        )

    def forward(self, images, meta=None):
        img_feats = self.backbone(images)
        if len(img_feats.shape) == 3:
            img_feats = img_feats.mean(dim=1)
            
        meta_feats = self.meta_mlp(meta) if meta is not None else self.meta_mlp(torch.zeros(images.size(0), 11, device=images.device))
        return self.head(torch.cat([img_feats, meta_feats], dim=1))

# ── Model Builders ──────────────────────────────────────────────────────────
def build_densenet121(pretrained=False):
    return MultiModalDenseNet(pretrained=pretrained)
def build_efficientnetv2s(pretrained=False):
    return MultiModalEfficientNet(pretrained=pretrained)
def build_swin_tiny(pretrained=False):
    return MultiModalSwin(pretrained=pretrained)
def build_convnext_tiny(pretrained=False):
    return MultiModalConvNeXt(pretrained=pretrained)
def build_resnet200d(pretrained=False):
    return MultiModalResNet(pretrained=pretrained)

def load_model(builder_fn, ckpt_path, device):
    model = builder_fn()
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    state_dict = ckpt.get('model_state_dict', ckpt)
    model.load_state_dict(state_dict)
    model.to(device).eval()
    return model

# ── Target Layer Finder ─────────────────────────────────────────────────────
def find_last_conv_layer(model, model_name):
    # Dynamically extract backbone if it's a multimodal wrapper
    base_model = getattr(model, 'backbone', model)
    
    if 'densenet' in model_name:
        return base_model.features.denseblock4
    elif 'efficientnet' in model_name:
        return base_model.conv_head
    elif 'swin' in model_name:
        if hasattr(base_model, 'layers'):
            return base_model.layers[-1].blocks[-1]
    elif 'convnext' in model_name:
        return base_model.stages[-1].blocks[-1]
    elif 'resnet' in model_name:
        return base_model.layer4[-1]
    return None

In [5]:
# ── CELL 3b : Multiclass (Severity) Model Architectures ──────────────────────
class TabularMLP(nn.Module):
    """Processes clinical metadata (age, sex, site) before fusion with image features."""
    def __init__(self, in_dim: int, hidden_dim: int = 64, out_dim: int = 32,
                 dropout: float = 0.2, use_gelu: bool = True):
        super().__init__()
        act = nn.GELU() if use_gelu else nn.ReLU(inplace=True)
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim), nn.BatchNorm1d(hidden_dim), act,
            nn.Dropout(dropout), nn.Linear(hidden_dim, out_dim), act,
        )
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

class MetadataSeverityModel(nn.Module):
    """DenseNet-121 and EfficientNetV2-S severity models."""
    def __init__(self, model_name: str, num_outputs: int, meta_dim: int,
                 pretrained: bool = False, drop_rate: float = 0.3,
                 drop_path_rate: float = 0.0, use_gelu: bool = True):
        super().__init__()
        kwargs = dict(pretrained=pretrained, num_classes=0, drop_rate=drop_rate)
        if drop_path_rate > 0:
            kwargs['drop_path_rate'] = drop_path_rate
        self.backbone  = timm.create_model(model_name, **kwargs)
        feat_dim       = self.backbone.num_features
        self.use_meta  = meta_dim > 0
        if self.use_meta:
            self.meta_mlp = TabularMLP(meta_dim, hidden_dim=64, out_dim=32, dropout=0.2, use_gelu=use_gelu)
            fusion_dim = feat_dim + 32
        else:
            fusion_dim = feat_dim
            
        norm = nn.BatchNorm1d(fusion_dim) if 'densenet' in model_name else nn.LayerNorm(fusion_dim)
        act  = nn.ReLU(inplace=True) if 'densenet' in model_name else nn.GELU()
        self.head = nn.Sequential(
            norm, nn.Linear(fusion_dim, 512 if 'densenet' in model_name else 256), act,
            nn.Dropout(drop_rate), nn.Linear(512 if 'densenet' in model_name else 256, num_outputs),
        )
        
    def forward(self, images: torch.Tensor, meta: torch.Tensor = None):
        feat = self.backbone(images)
        if self.use_meta and meta is not None:
            feat = torch.cat([feat, self.meta_mlp(meta)], dim=1)
        return self.head(feat)

class SwinSeverityModel(nn.Module):
    """Swin-Tiny severity model with LayerNorm head and metadata fusion."""
    def __init__(self, model_name: str, num_outputs: int, meta_dim: int,
                 pretrained: bool = False, drop_rate: float = 0.2,
                 drop_path_rate: float = 0.2):
        super().__init__()
        self.backbone = timm.create_model(
            model_name, pretrained=pretrained,
            num_classes=0, drop_rate=drop_rate, drop_path_rate=drop_path_rate
        )
        feat_dim      = self.backbone.num_features 
        self.use_meta = meta_dim > 0
        if self.use_meta:
            self.meta_mlp = TabularMLP(meta_dim, 64, 32, 0.2, use_gelu=True)
            fusion_dim = feat_dim + 32
        else:
            fusion_dim = feat_dim
            
        self.head = nn.Sequential(
            nn.LayerNorm(fusion_dim), nn.Linear(fusion_dim, 512), nn.GELU(),
            nn.Dropout(0.3), nn.Linear(512, 128), nn.GELU(),
            nn.Dropout(0.1), nn.Linear(128, num_outputs),
        )
        
    def forward(self, images: torch.Tensor, meta: torch.Tensor = None):
        feat = self.backbone(images)
        if self.use_meta and meta is not None:
            # FIXED: Corrected self.meta_mlp(feat) to self.meta_mlp(meta)
            feat = torch.cat([feat, self.meta_mlp(meta)], dim=1) 
        return self.head(feat)

# ── Severity model builders ────────────────────────────────────────────────────
def build_severity_densenet(meta_dim: int = SEVERITY_META_DIM, pretrained: bool = False):
    return MetadataSeverityModel(
        'densenet121', SEVERITY_NUM_OUTPUTS, meta_dim,
        pretrained=pretrained, drop_rate=0.3, drop_path_rate=0.0, use_gelu=False
    )
def build_severity_efficientnet(meta_dim: int = SEVERITY_META_DIM, pretrained: bool = False):
    return MetadataSeverityModel(
        'tf_efficientnetv2_s', SEVERITY_NUM_OUTPUTS, meta_dim,
        pretrained=pretrained, drop_rate=0.3, drop_path_rate=0.2, use_gelu=True
    )
def build_severity_swin(meta_dim: int = SEVERITY_META_DIM, pretrained: bool = False):
    return SwinSeverityModel(
        'swin_tiny_patch4_window7_224', SEVERITY_NUM_OUTPUTS, meta_dim,
        pretrained=pretrained, drop_rate=0.2, drop_path_rate=0.2
    )
    
def load_severity_model(builder_fn, ckpt_path, device, meta_dim=SEVERITY_META_DIM):
    model = builder_fn(meta_dim=meta_dim)
    ckpt  = torch.load(ckpt_path, map_location=device, weights_only=False)
    state = ckpt.get('model_state', ckpt.get('model_state_dict', ckpt))
    model.load_state_dict(state, strict=False)
    model.to(device).eval()
    return model

def coral_logits_to_label(logits: torch.Tensor) -> torch.Tensor:
    return (torch.sigmoid(logits) > 0.5).sum(dim=1)

def find_severity_target_layer(model, model_type: str):
    if model_type == 'densenet':
        return model.backbone.features.denseblock4
    elif model_type == 'efficientnet':
        return model.backbone.conv_head
    elif model_type == 'swin':
        return model.backbone.layers[-1].blocks[-1]
    return None

print('Severity model architectures loaded.')
print(f'  Classes : MetadataSeverityModel (DenseNet, EfficientNet), SwinSeverityModel')
print(f'  Builders: build_severity_densenet | build_severity_efficientnet | build_severity_swin')
print(f'  CORAL   : {SEVERITY_NUM_OUTPUTS} outputs -> severity class 0/1/2/3 (AKIEC/BCC/SCC/MEL)')

Severity model architectures loaded.
  Classes : MetadataSeverityModel (DenseNet, EfficientNet), SwinSeverityModel
  Builders: build_severity_densenet | build_severity_efficientnet | build_severity_swin
  CORAL   : 3 outputs -> severity class 0/1/2/3 (AKIEC/BCC/SCC/MEL)


In [6]:
# ── CELL 4 : Dataset ──────────────────────────────────────────────────────────
class XAIDataset(Dataset):
    def __init__(self, df: pd.DataFrame, transform):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row['image_path']).convert('RGB')
        img_t = self.transform(img)
        return img_t, str(row.get('image_id', idx))

def get_transforms(img_size):
    return T.Compose([
        T.Resize((img_size, img_size)), T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
    ])

val_df = pd.read_csv(f'{INPUT_CSV_DIR}/binary_val.csv')
test_df = pd.read_csv(f'{INPUT_CSV_DIR}/binary_test.csv')
eval_df = pd.concat([val_df, test_df], ignore_index=True)
eval_df['image_path'] = eval_df['image_path'].str.replace(OLD_PATH, NEW_PATH, regex=False)

In [ ]:
# ── Swin-Tiny Binary Setup ──
import os
import gc
import cv2
import torch
import numpy as np
from tqdm import tqdm
from torch.utils.data import DataLoader
from pytorch_grad_cam import EigenCAM

class SwinWrapper(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
    def forward(self, x):
        return self.model(x, meta=None)

def reshape_transform(tensor):
    if tensor.ndim == 4:
        return tensor.permute(0, 3, 1, 2)
    elif tensor.ndim == 3:
        B, L, C = tensor.shape
        H = int(np.sqrt(L))
        return tensor.reshape(B, H, H, C).permute(0, 3, 1, 2)
    return tensor

print("[1/1] Loading Swin-Tiny model...")
swin_model = load_model(build_swin_tiny, SWIN_CKPT, DEVICE)
target_layers = [swin_model.backbone.layers[-1].blocks[-1]]

SWIN_OUT_DIR = os.path.join(BINARY_OUT, 'heatmaps_swin')
os.makedirs(SWIN_OUT_DIR, exist_ok=True)

already_done = {f.replace('.npy', '') for f in os.listdir(SWIN_OUT_DIR) if f.endswith('.npy')}
print(f"Already processed: {len(already_done)} files")

loader = DataLoader(
    XAIDataset(eval_df, get_transforms(IMG_SIZE_SWIN)),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True
)

wrapped_model = SwinWrapper(swin_model)
cam = EigenCAM(model=wrapped_model, target_layers=target_layers, reshape_transform=reshape_transform)



In [ ]:
# ── TEST: Generate and Display First 10 Heatmaps ──
import matplotlib.pyplot as plt
from pytorch_grad_cam.utils.image import show_cam_on_image
import numpy as np

print("Generating 10 preview heatmaps...")

fig, axes = plt.subplots(2, 5, figsize=(20, 8))
axes = axes.flatten()

for i, (imgs, img_ids) in enumerate(loader):
    if i >= 10: break
    image_id = img_ids[0]
    img_tensor = imgs.to(DEVICE)
    
    hmap = cam(input_tensor=img_tensor)[0, :]
    
    img_disp = imgs[0].permute(1, 2, 0).cpu().numpy()
    img_disp = (img_disp * np.array([0.229, 0.224, 0.225])) + np.array([0.485, 0.456, 0.406])
    img_disp = np.clip(img_disp, 0, 1)
    
    vis = show_cam_on_image(img_disp, hmap, use_rgb=True)
    
    axes[i].imshow(vis)
    axes[i].set_title(f"Swin-Tiny: {image_id}")
    axes[i].axis('off')

plt.tight_layout()
plt.show()
print("If these look correct, you can delete this cell and run the next one!")



In [ ]:
# ── MAIN: Generate All Heatmaps ──
skipped = 0
processed = 0
errors = 0

for imgs, img_ids in tqdm(loader, desc="Binary Swin EigenCAM"):
    image_id = img_ids[0]
    
    if image_id in already_done:
        skipped += 1
        continue
        
    imgs = imgs.to(DEVICE)
    
    try:
        hmap = cam(input_tensor=imgs)[0, :]
        np.save(os.path.join(SWIN_OUT_DIR, f'{image_id}.npy'), hmap)
        processed += 1
    except Exception as e:
        print(f"Failed {image_id}: {e}")
        errors += 1

del swin_model
gc.collect()
torch.cuda.empty_cache()

print("Swin Binary Heatmaps complete!")
print(f"Processed: {processed}")
print(f"Skipped: {skipped}")
print(f"Errors: {errors}")

